### Load and Clip the Arctic DEM

Notebook contents 
* Take a look at the Arctic DEM downloaded in `hdd/snow_hydrology/DEMs/`
* Develop processing workflow 

created by Cassie Lumbrazo\
last updated: April 2025\
run location: UAS linux\
python environment: **rasterio**

In [1]:
# import packages 
%matplotlib inline

# plotting packages 
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns 

sns.set_theme()
# plt.rcParams['figure.figsize'] = [12,6] #overriding size

# data packages 
import pandas as pd
import numpy as np
import xarray as xr
from datetime import datetime

import scipy

from matplotlib.patches import Patch
from matplotlib.colors import ListedColormap
from matplotlib import ticker

import rioxarray
import rasterio 
import cfgrib
import os

from rasterio.warp import transform_bounds

In [2]:
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.mask import mask
from shapely.geometry import box, mapping
from pyproj import Transformer, CRS
import os

In [3]:
pwd

'/home/cassie/python/repos/snow_model_forcing/large_juneau_domain'

### Open the ArcticDEM, which is already in UTM 

In [4]:
# File paths 
filepath = '/hdd/snow_hydrology/DEMs/ArcticDEM/'
# filename =  '40_05_2_2_2m_v4.1_dem.tif' # this file is in EPSG:3413 
filename =  '40_05_2_2_2m_v4.1_dem_UTM.tif'# now, this file should be in UTM coordinates (EPSG:32608)
raster_path = os.path.join(filepath, filename)

# open the DEM with rioxarray
dem = rioxarray.open_rasterio(raster_path, masked=True)
# dem = dem.rio.write_crs("EPSG:32608")  # Set the CRS to UTM Zone 8N

dem.rio.crs

CRS.from_wkt('PROJCS["WGS 84 / UTM zone 8N",GEOGCS["WGS 84",DATUM["WGS_1984",SPHEROID["WGS 84",6378137,298.257223563,AUTHORITY["EPSG","7030"]],AUTHORITY["EPSG","6326"]],PRIMEM["Greenwich",0,AUTHORITY["EPSG","8901"]],UNIT["degree",0.0174532925199433,AUTHORITY["EPSG","9122"]],AUTHORITY["EPSG","4326"]],PROJECTION["Transverse_Mercator"],PARAMETER["latitude_of_origin",0],PARAMETER["central_meridian",-135],PARAMETER["scale_factor",0.9996],PARAMETER["false_easting",500000],PARAMETER["false_northing",0],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH],AUTHORITY["EPSG","32608"]]')

Or just using rasterio.open directly, 

In [5]:
with rasterio.open(raster_path) as src:
    print("Original CRS:", src.crs)

src

Original CRS: EPSG:32608


<closed DatasetReader name='/hdd/snow_hydrology/DEMs/ArcticDEM/40_05_2_2_2m_v4.1_dem_UTM.tif' mode='r'>

## Clip the DEM and Save 

 The bounding box is currently in lat/lon (EPSG:4326). Thus, to clip using UTM coordinates, then we need to provide the bounding box in the correct UTM zone

 `(e.g., UTM Zone 8N is common for SE Alaska → EPSG:32608).`

The "Large Juneau Area" Main sites bounding box

-135.0734:-134.0303 58.1539:58.8081

In [14]:
import rasterio
from rasterio.mask import mask
from rasterio.warp import calculate_default_transform, reproject, Resampling
from shapely.geometry import box
# from shapely.ops import mapping
# from shapely import mapping  # Updated import for Shapely 2.x
from shapely.geometry import box, mapping  # Correct import for mapping

from pyproj import CRS, Transformer
import numpy as np
import shutil
from osgeo import gdal

In [ ]:
# Define clipping bounds in lon/lat (WGS84) - Large Juneau Domain
xmin, xmax = -135.0734, -134.0303
ymin, ymax = 58.1539, 58.8081

# Transform bounds to UTM Zone 8N (EPSG:32608)
utm_crs = CRS.from_epsg(32608)
transformer = Transformer.from_crs("EPSG:4326", utm_crs, always_xy=True)
xmin_utm, ymin_utm = transformer.transform(xmin, ymin)
xmax_utm, ymax_utm = transformer.transform(xmax, ymax)

# Create a UTM bounding box polygon for clipping
utm_bbox = box(xmin_utm, ymin_utm, xmax_utm, ymax_utm)
clip_polygon = [mapping(utm_bbox)]

# Input and output paths
raster_path = "/hdd/snow_hydrology/DEMs/ArcticDEM_edits/40_05_2_2_2m_v4.1_dem_UTM.tif"
clip_raster_path = '/hdd/snow_hydrology/DEMs/ArcticDEM_edits/large_juneau_domain/dem_2m_UTM_clip_large_juneau_domain.tif'

# Target resolution in meters (square pixels)
target_res = 2

# Clip and resample the raster
with rasterio.open(raster_path) as src:
    # Clip the raster to the bounding box
    clipped, clipped_transform = mask(src, clip_polygon, crop=True)
    
    # Calculate the transform for square pixels at target resolution
    dst_transform, dst_width, dst_height = calculate_default_transform(
        src.crs, src.crs,  # No reprojection, just resampling
        src.width, src.height,
        *utm_bbox.bounds,
        resolution=(target_res, target_res)
    )
    
    # Update metadata for the output raster
    kwargs = src.meta.copy()
    kwargs.update({
        "height": dst_height,
        "width": dst_width,
        "transform": dst_transform,
        "dtype": src.dtypes[0],
        "crs": src.crs
    })
    
    # Write the clipped and resampled raster
    with rasterio.open(clip_raster_path, "w", **kwargs) as dst:
        for i in range(1, src.count + 1):
            reproject(
                source=clipped[i-1],
                destination=rasterio.band(dst, i),
                src_transform=clipped_transform,
                src_crs=src.crs,
                dst_transform=dst_transform,
                dst_crs=src.crs,
                resampling=Resampling.bilinear
            )

print(f"Clipped and resampled raster saved to: {clip_raster_path}")

# Check for NaNs in the clipped DEM
with rasterio.open(clip_raster_path) as src:
    data = src.read(1)  # Read the first band
    nan_mask = np.isnan(data)
    nan_count = np.sum(nan_mask)
    
    if nan_count > 0:
        print(f"Found {nan_count} NaN values in the DEM.")
        # Optionally, print locations (row, col) of NaNs
        nan_indices = np.where(nan_mask)
        print(f"NaN locations (first 10): {list(zip(nan_indices[0][:10], nan_indices[1][:10]))}")
    else:
        print("No NaN values found in the DEM.")


# Fill NaNs using GDAL
src_path = clip_raster_path
dst_path = "/hdd/snow_hydrology/DEMs/ArcticDEM_edits/large_juneau_domain/dem_2m_UTM_clip_large_juneau_domain_filled_gdal.tif"

# Copy the original raster to the new path
shutil.copy(src_path, dst_path)

# Enable GDAL exceptions to suppress the warning
gdal.UseExceptions()

# Open the copy in update mode
ds = gdal.Open(dst_path, gdal.GA_Update)
band = ds.GetRasterBand(1)

# Perform nodata filling (maxSearchDist=5, smoothingIterations=0)
gdal.FillNodata(targetBand=band, maskBand=None, maxSearchDist=5, smoothingIterations=0)

# Clean up
band = None
ds = None

print(f"Filled raster saved to: {dst_path}")

Clipped and resampled raster saved to: /hdd/snow_hydrology/DEMs/ArcticDEM_edits/large_juneau_domain/dem_2m_UTM_clip_large_juneau_domain.tif
No NaN values found in the DEM.


/home/cassie/programs/miniforge3/envs/rasterio/lib/python3.13/site-packages/osgeo/gdal.py:311: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Filled raster saved to: /hdd/snow_hydrology/DEMs/ArcticDEM_edits/large_juneau_domain/dem_2m_UTM_clip_large_juneau_domain_filled_gdal.tif


In [ ]:
# Plot the filled DEM
with rasterio.open(dst_path) as src:
    dem_data = src.read(1)  # Read the first band
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]  # UTM bounds for plotting

    plt.figure(figsize=(10, 8))
    plt.imshow(dem_data, extent=extent, cmap='terrain', origin='upper')
    plt.colorbar(label='Elevation (m)')
    plt.title('Filled DEM - Large Juneau Domain (UTM Zone 8N)')
    plt.xlabel('Easting (m)')
    plt.ylabel('Northing (m)')
    plt.grid(True, alpha=0.3)
